# サーバ証明書の取得方式の変更

---

構築したCoursewareHubのサーバ証明書の取得方式を、ACME による自動取得 (方式A) と証明書ファイルの手動配置 (方式B) の間で変更します。

## 概要

「121-CoursewareHubのセットアップ-ローカルユーザ認証」で選択したサーバ証明書の取得方式 (`use_acme`) を変更します。現在の方式が方式A であれば方式B に、方式B であれば方式A に変更します。

- 方式B から方式A に変更する場合: ACME サーバのパラメータを指定し、auth-proxy コンテナが起動時に ACME サーバから証明書を取得します
- 方式A から方式B に変更する場合: 新しいサーバ証明書と秘密鍵のファイルを配置し、それを利用するように変更します

取得方式の変更には、サーバ証明書の配置だけでなく以下の操作が必要となります。

- CoursewareHub のコンテナを停止して起動しなおす: ACME の設定は Docker の secret として auth-proxy コンテナに渡しています。secret の追加や削除を反映させるため、一度停止して作りなおします
- registry の CA 証明書を更新する: Let's Encrypt の staging を利用する場合はテスト用 CA の証明書を各ノードの docker と jupyterhub コンテナに信頼させ、それ以外の場合は削除します
- `docker-compose.yml` を生成しなおす: 取得方式によって、コンテナにマウントする証明書のディレクトリなどが変わります

> **注意:** このNotebookの操作では CoursewareHub のコンテナを一度停止するため、操作中は CoursewareHub を利用できません。

> **学認連携を設定している場合:** 現在のところ、学認連携を設定した CoursewareHub の取得方式の変更には対応していません。方式A の CoursewareHub には学認連携を設定できないためです。

同じ取得方式のまま証明書を更新する場合は、以下のNotebookを利用してください。

- 方式B の証明書ファイルの更新: 「741-サーバ証明書の更新」
- 方式A の ACME サーバの切り替え: 「742-ACMEサーバの切り替え」

### グループ名

更新対象となるCoursewareHubを構築したときに指定したansibleのグループ名を指定します。

ansibleのグループ名を確認するために `group_vars`ファイル名の一覧を表示します。

In [ ]:
!ls -1 --hide all group_vars/

上のセルの出力結果を参考にしてグループ名を次のセルに指定してください。

In [ ]:
# (例)
# ugroup_name = 'CoursewareHub'

ugroup_name = 

### チェック

指定されたグループ名が前提条件を満たしていることを確認します。

Ansibleの設定ファイルの場所を環境変数に設定しておきます。

In [ ]:
from pathlib import Path
import os

cfg_ansible = Path('ansible.cfg')
if cfg_ansible.exists():
    os.environ['ANSIBLE_CONFIG'] = str(cfg_ansible.resolve())

構築対象となるVCノードにアクセスできることを確認します。

In [ ]:
target_hub = f'{ugroup_name}_manager'

!ansible {target_hub} -m ping

UnitGroup名に対応する group_varsファイルが存在していることを確認します。

In [ ]:
!test -f group_vars/{ugroup_name}

CoursewareHubのサービスが実行されていることを確認します。

In [ ]:
cmd = "docker stack services --format '{{.Replicas}}' coursewarehub | grep '1/1' | wc -l"
!ansible {target_hub} -m shell -a "{{% raw %}} [ \$({cmd}) -eq 4 ] {{% endraw %}}"

group_varsから現在の設定値を読み込みます。

In [ ]:
import yaml
from pathlib import Path

gvars_path = Path(f'group_vars/{ugroup_name}')
with gvars_path.open() as f:
    gvars = yaml.safe_load(f)

学認連携が設定されていないことを確認し、現在の取得方式と変更後の取得方式を表示します。

In [ ]:
if gvars.get('enable_federation', False) or gvars.get('auth_fqdn'):
    raise RuntimeError('学認連携を設定した CoursewareHub の取得方式の変更には対応していません。')

method_names = {
    True: '方式A (ACME による自動取得)',
    False: '方式B (証明書ファイルの手動配置)',
}
use_acme = not bool(gvars.get('use_acme', False))
certbot_data_dir = gvars.get('certbot_data_dir', '/jupyter/certbot')

print(f'取得方式: {method_names[not use_acme]} -> {method_names[use_acme]}')

## ACME のパラメータの指定 (方式A に変更する場合)

ACME サーバから証明書を取得するためのパラメータを指定します。方式B に変更する場合、この章のセルは何も行いません。各パラメータの詳細は「121-CoursewareHubのセットアップ-ローカルユーザ認証」の「方式A: ACME による自動取得」を参照してください。

> サーバ証明書の取得には HTTP-01 チャレンジを使用するため、対象 FQDN のポート 80 (HTTP) が ACME サーバからインターネット経由で到達可能である必要があります。

### ACME サーバ

利用する ACME サーバの directory URL を `acme_server` に指定します。主な ACME サーバの URL を以下に示します。

| 利用する CA | directory URL | EAB |
| --- | --- | --- |
| Let's Encrypt (テスト / staging) | `https://acme-staging-v02.api.letsencrypt.org/directory` | 不要 |
| Let's Encrypt (本番) | `https://acme-v02.api.letsencrypt.org/directory` | 不要 |
| UPKI 電子証明書発行サービス (NII) | UPKI 運用窓口から提供される URL | 必要 |

Let's Encrypt を利用する場合は、まず staging で動作確認を行い、その後「742-ACMEサーバの切り替え」で本番に切り替える手順を想定しています。

In [ ]:
# (例)
# acme_server = 'https://acme-staging-v02.api.letsencrypt.org/directory'   # Let's Encrypt (staging)
# acme_server = 'https://acme-v02.api.letsencrypt.org/directory'           # Let's Encrypt (本番)
# acme_server = 'UPKI 運用窓口から提供された directory URL'                  # UPKI 電子証明書発行サービス

acme_server = ''

### 連絡先メールアドレス

ACME サーバへのアカウント登録に用いるメールアドレスを `acme_email` に指定します。証明書の失効などに関する連絡先となります。省略することはできません。

In [ ]:
# (例)
# acme_email = 'noc@example.ac.jp'

acme_email = gvars.get('acme_email', '')

### 鍵種別

証明書の鍵種別 (`rsa` または `ecdsa`) を `acme_key_type` に指定します。

In [ ]:
# (例)
# acme_key_type = 'rsa'
# acme_key_type = 'ecdsa'

acme_key_type = gvars.get('acme_key_type', 'rsa')

### External Account Binding (EAB)

CA が EAB を要求する場合は、次のセルを実行して表示される入力欄に Key ID (`acme_eab_kid`)、HMAC 鍵 (`acme_eab_hmac_key`) の順に入力してください。機密情報のため、入力した値は画面に表示されません。

- UPKI 電子証明書発行サービス: UPKI の管理ポータルで発行された値を入力してください
- Let's Encrypt: EAB は不要です。何も入力せずに Enter キーを押してください

In [ ]:
from getpass import getpass

if use_acme:
    acme_eab_kid = getpass('ACME EAB KID: ')
    acme_eab_hmac_key = getpass('ACME HMAC Key: ')

### 設定値の確認

指定した値を表示して確認します。EAB の Key ID と HMAC 鍵は値を表示せず、入力の有無のみを表示します。

In [ ]:
if use_acme:
    if len(acme_server) == 0:
        raise RuntimeError('acme_server を指定してください。')
    if len(acme_email) == 0:
        raise RuntimeError('acme_email を指定してください。')
    print(f'ACME Server: {acme_server}')
    print(f'ACME Email: {acme_email}')
    print(f'ACME Key type: {acme_key_type}')
    print(f'EAB KID: {"入力あり" if acme_eab_kid else "なし"}')
    print(f'EAB HMAC Key: {"入力あり" if acme_eab_hmac_key else "なし"}')

### 事前確認

ACME サーバによる検証の失敗はレート制限の対象として数えられます。コンテナを停止する前に、次のセルに表示されるホスト名と HTTP-01 を [Let's Debug](https://letsdebug.net/) に指定して、問題が検出されないことを確認してください。Let's Debug は、複数の地点からの DNS の名前解決やポート 80 への到達性などを確認するサービスです。

In [ ]:
if use_acme:
    print(gvars['master_fqdn'])

## 証明書ファイルの配置 (方式B に変更する場合)

auth-proxyコンテナで使用するサーバ証明書、秘密鍵などのファイルを配置します。方式A に変更する場合、この章のセルは何も行いません。

出どころなどの情報を必要以上に残さないためにNotebookからの操作ではなく、ターミナルなどから **managerノードに ssh でログインして操作を行ってください**。

manager ノードのIPアドレスを確認します。表示されたIPアドレスに対して、ユーザ名`vcp`と「VCノード作成」のNotebookで設定したSSHの秘密鍵を指定することで manager ノードにsshでログインできます。

In [ ]:
if not use_acme:
    !ansible {target_hub} -c local -m debug -a 'var=servicenet_ip'

サーバ証明書、秘密鍵をそれぞれ以下のセルの実行結果に表示されるパスに配置してください。

In [ ]:
if not use_acme:
    !ansible {target_hub} -c local -m debug -a 'msg="/home/{{{{ansible_user}}}}/certs/{{{{master_fqdn}}}}.cer"'

In [ ]:
if not use_acme:
    !ansible {target_hub} -c local -m debug -a 'msg="/home/{{{{ansible_user}}}}/certs/{{{{master_fqdn}}}}.key"'

中間CA証明書がある場合は、サーバ証明書と同じディレクトリに配置してください。中間CA証明書を配置した場合は次のセルにそのファイル名を指定してください。

In [ ]:
# 中間CA証明書を指定する場合は文字列を代入する
# (例)
# intermediate_certfile = 'nii-odca4g7rsa.cer'

intermediate_certfile = None

また個別の中間CA証明書を配置するのではなく、中間CA証明書を連結したファイルを配置することもできます。その場合は次のセルの実行結果に表示されるパスに配置してください。中間CA証明書を連結したファイルが存在している場合は、個別の中間CA証明書を指定しても**連結したファイルの方が優先**されます。

In [ ]:
if not use_acme:
    !ansible {target_hub} -c local -m debug -a 'msg="/home/{{{{ansible_user}}}}/certs/{{{{master_fqdn}}}}.chained.cer"'

**証明書などの配置を行った後に、これ以降の操作を行ってください。**

証明書が配置されたことを確認します。managerノードに配置されたサーバ証明書の内容を表示してみます。

In [ ]:
if not use_acme:
    !ansible {target_hub} -a \
        'openssl x509 -noout -text -in certs/{{{{master_fqdn}}}}.cer'

秘密鍵の内容を表示してみます。

In [ ]:
if not use_acme:
    !ansible {target_hub} -m shell -a \
        'openssl ec -noout -text -in  certs/{{{{master_fqdn}}}}.key || \
        openssl rsa -noout -text -in  certs/{{{{master_fqdn}}}}.key'

中間CA証明書を指定した場合は、そのファイルがmanagerノードに配置されていることを確認します。

In [ ]:
if not use_acme and intermediate_certfile:
    !ansible {target_hub} -a \
        'openssl x509 -noout -text -in certs/{intermediate_certfile}'

中間CA証明書を連結したファイルを配置した場合は、その内容を確認します。

In [ ]:
if not use_acme:
    !ansible {target_hub} -m shell -a 'chdir=certs \
        test ! -f {{{{master_fqdn}}}}.chained.cer || \
        ( openssl crl2pkcs7 -nocrl -certfile {{{{master_fqdn}}}}.chained.cer | openssl pkcs7 -print_certs -noout )'

## 設定の保存

変更後の取得方式 (`use_acme`) と、方式に応じたパラメータを `group_vars` に保存します。方式A の場合は ACME のパラメータを保存し、方式B の場合は ACME のパラメータを削除して中間CA証明書のファイル名 (指定した場合) を保存します。

In [ ]:
if use_acme:
    gvars.update({
        'use_acme': True,
        'acme_server': acme_server,
        'acme_email': acme_email,
        'acme_key_type': acme_key_type,
        'acme_eab_kid': acme_eab_kid,
        'acme_eab_hmac_key': acme_eab_hmac_key,
    })
    gvars.pop('intermediate_certfile', None)
else:
    gvars['use_acme'] = False
    for k in ('acme_server', 'acme_email', 'acme_key_type',
              'acme_eab_kid', 'acme_eab_hmac_key'):
        gvars.pop(k, None)
    if intermediate_certfile:
        gvars['intermediate_certfile'] = intermediate_certfile
    else:
        gvars.pop('intermediate_certfile', None)

with gvars_path.open(mode='w') as f:
    yaml.safe_dump(gvars, stream=f)

## コンテナの停止

CoursewareHub のコンテナを停止します。`--detach=false` を指定しているので、全てのサービスが停止するまで待ち合わせを行います。待ち合わせは最大10分で打ち切ります。

In [ ]:
!ansible {target_hub} -a 'timeout 600 docker stack rm --detach=false coursewarehub'

## サーバ証明書と ACME 設定ファイルの配置

変更後の取得方式に応じて、以下のファイルを配置します。

- 方式A: 証明書ファイルの手動配置で利用していたサーバ証明書と秘密鍵 (`/etc/jupyterhub/nginx/certs/server.cer`、`server.key`) を削除し、ACME 設定ファイル (`/opt/coursewarehub/acme-config.sh`) と certbot のディレクトリを配置します。auth-proxy コンテナはサーバ証明書のファイルが存在しない場合に ACME による自動取得を行います
- 方式B: managerノードに配置したサーバ証明書と秘密鍵を、`/etc/jupyterhub/nginx/certs/` などのコンテナが参照するディレクトリにコピーします

まずはチェックモードで確認を行います。

In [ ]:
!ansible-playbook -l {target_hub} -CDv playbooks/setup-certs.yml || true

実際に配置を行います。

In [ ]:
!ansible-playbook -l {target_hub} -v playbooks/setup-certs.yml

## 取得済みの証明書の削除 (方式A に変更する場合)

以前に ACME を利用していた場合は、そのときに取得した証明書と更新設定が残っています。auth-proxy コンテナは証明書が存在しない場合にのみ新規取得を行うため、残っている証明書と更新設定を削除します。ACME サーバのアカウント情報 (`accounts/`) は ACME サーバごとに分かれているため削除しません。

方式B に変更する場合、この章のセルは何も行いません。certbot のディレクトリは方式B ではコンテナにマウントされないため、残したままで構いません。

In [ ]:
if use_acme:
    !ansible {target_hub} -b -m shell -a \
        'rm -rf {certbot_data_dir}/live/{{{{master_fqdn}}}} \
                {certbot_data_dir}/archive/{{{{master_fqdn}}}} \
                {certbot_data_dir}/renewal/{{{{master_fqdn}}}}.conf'

## registry の CA 証明書の更新

変更後の取得方式と ACME サーバに応じて、各ノードの docker と jupyterhub コンテナに信頼させる CA 証明書を更新します。Let's Encrypt の staging を利用する場合はテスト用 CA の証明書を配置し、それ以外の場合は配置済みのものを削除します。

jupyterhub コンテナに信頼させるためのファイルはコンテナの起動時に参照されるため、コンテナを起動する前に実行します。また registry からイメージを pull する worker ノードでも必要となるため、managerノードとworkerノードの全てに対して実行します。まずはチェックモードで確認を行います。

In [ ]:
!ansible-playbook -l {ugroup_name} -CDv playbooks/deploy-registry-ca.yml || true

実際に CA 証明書の更新を行います。

In [ ]:
!ansible-playbook -l {ugroup_name} -v playbooks/deploy-registry-ca.yml

## docker-compose.yml の更新

変更後の取得方式に応じて `docker-compose.yml` を生成しなおします。まずはチェックモードで確認を行います。

In [ ]:
!ansible-playbook -l {target_hub} -CDv playbooks/deploy-docker-compose.yml || true

実際に `docker-compose.yml` の更新を行います。

In [ ]:
!ansible-playbook -l {target_hub} playbooks/deploy-docker-compose.yml

## コンテナの起動

コンテナを起動します。方式A の場合、auth-proxy コンテナは起動時に ACME サーバから証明書を取得します。

`--detach=false` を指定しているので、全てのサービスのコンテナが起動するまで待ち合わせを行います。待ち合わせは最大10分で打ち切ります。

> 次のセルの実行結果がエラーとなる場合は、その後のセルを実行してコンテナの実行状況やログを確認してください。

In [ ]:
!ansible {target_hub} -a 'chdir={{{{compose_dir}}}} \
    timeout 600 docker stack deploy --detach=false -c docker-compose.yml coursewarehub'

`auth-proxy` コンテナのログを表示してみます。

> 証明書の取得の失敗や証明書の設定の誤りなどのエラーが表示されていないことを確認してください。

In [ ]:
!ansible {target_hub} -a 'docker service logs --since=30m coursewarehub_auth-proxy'

**証明書の取得に失敗した場合** (方式A に変更した場合)

auth-proxy コンテナは ACME による証明書の取得に失敗すると終了し、15分後に再起動されて取得をやりなおします。原因を取り除かないまま再試行を続けると ACME サーバのレート制限 (Let's Encrypt の本番では、同じホスト名に対する認証の失敗は1時間に5回まで) に達し、しばらく証明書を取得できなくなります。

上のログに証明書の取得の失敗 (`urn:ietf:params:acme:error` など) が表示されている場合は、次のセルのコメントを外して auth-proxy コンテナを停止してから、以下の方法で原因を調べてください。

- certbot のログ: managerノードの `/jupyter/certbot-logs/letsencrypt.log`
- [Let's Debug](https://letsdebug.net/) に `master_fqdn` と HTTP-01 を指定して、ACME サーバから見た DNS の名前解決とポート 80 への到達性を確認する

原因を取り除いた後に、auth-proxy コンテナを再開してください。

In [ ]:
# 証明書の取得に失敗した場合に auth-proxy コンテナを停止する
# !ansible {target_hub} -a 'docker service scale coursewarehub_auth-proxy=0'

# 原因を取り除いた後に auth-proxy コンテナを再開する
# !ansible {target_hub} -a 'docker service scale coursewarehub_auth-proxy=1'

`registry` コンテナのログを表示してみます。

In [ ]:
!ansible {target_hub} -a 'docker service logs --since=30m coursewarehub_registry'

## 証明書の確認

auth-proxy が 443 番と 5000 番 (registry) で新しい証明書を返していることを確認します。発行者 (`issuer`) が変更後の証明書の発行元になっていることを確認してください。

In [ ]:
for port in [443, 5000]:
    !ansible {target_hub} -m shell -a \
        'openssl s_client -connect {{{{master_fqdn}}}}:{port} -servername {{{{master_fqdn}}}} < /dev/null 2> /dev/null \
        | openssl x509 -noout -subject -issuer -dates'

全てのノードの docker から registry にログインできることを確認します。

In [ ]:
!ansible {ugroup_name} -b -m shell -a \
    'echo {{{{registry_pass}}}} | docker login -u cwh --password-stdin {{{{master_fqdn}}}}:5000'

## CoursewareHubにアクセスする

取得方式を変更したCoursewareHub環境にアクセスして正しく動作していることを確認してください。ログイン後にサーバの起動画面と Environments 画面が表示されることも確認してください。これらの画面は jupyterhub コンテナから registry に接続してコースイメージの一覧を取得します。

次のセルを実行するとCoursewareHubのアドレスを表示します。

In [ ]:
print(f'https://{gvars["master_fqdn"]}')